In [0]:
%run ../../00_common/data_utils

In [0]:
TOPIC_MAPPING = {
    "AUS|cbr":       "ConsumerBestRecordTopic,ConsumerBestRecordTopic_ALL",
    "HKG|cbr":       "CBR_HK,CBR_HK_ALL",
    "IDN|cbr":       "CBR_ID,CBR_ID_ALL",
    "JPN|cbr":       "CBR_JP,CBR_JP_ALL",
    "KOR|cbr":       "CBR_KR,CBR_KR_ALL",
    "MYS|cbr":       "CBR_MY,CBR_MY_ALL",
    "NZL|cbr":       "CBR_NZ,CBR_NZ_ALL",
    "PHL|cbr":       "CBR_PH,CBR_PH_ALL",
    "SGP|cbr":       "CBR_SG,CBR_SG_ALL",
    "THA|cbr":       "CBR_TH,CBR_TH_ALL",
    "TWN|cbr":       "CBR_TW,CBR_TW_ALL",
    "VNM|cbr":       "CBR_VN,CBR_VN_ALL",
    "AUS|cbrpublic": "CBRPublicTopic",
    "HKG|cbrpublic": "CBRPublic_HK",
    "IDN|cbrpublic": "CBRPublic_ID",
    "JPN|cbrpublic": "CBRPublic_JP",
    "KOR|cbrpublic": "CBRPublic_KR",
    "MYS|cbrpublic": "CBRPublic_MY",
    "NZL|cbrpublic": "CBRPublic_NZ",
    "PHL|cbrpublic": "CBRPublic_PH",
    "SGP|cbrpublic": "CBRPublic_SG",
    "THA|cbrpublic": "CBRPublic_TH",
    "TWN|cbrpublic": "CBRPublic_TW",
    "VNM|cbrpublic": "CBRPublic_VN",
    "KOR|cbrdj":     "CBR_DrJart_KR",
}

mapping_expr = F.create_map([F.lit(x) for pair in TOPIC_MAPPING.items() for x in pair])

In [0]:
def retry_backup_cbr_to_kafka(market_code, mdm_key):
    """
    根据 MarketCode + MDMKey 查询 t_cbr_dataset_backup，
    将每个 cbr_type 对应的 FinalJSON 发送到对应 Kafka topic。
    """
    market_code = market_code.strip().upper() if market_code else ""
    mdm_key = mdm_key.strip() if mdm_key else ""

    if not market_code:
        raise ValueError("MarketCode is required")
    if not mdm_key:
        raise ValueError("MDMKey is required")

    target_db = get_env_config('silver_mdm_anonymization_database')
    backup_table = f"{target_db}.t_cbr_dataset_backup"

    backup_df = (
        spark.table(backup_table)
        .where(
            (F.col("MarketCode") == F.lit(market_code)) &
            (F.col("MDMKey") == F.lit(mdm_key))
        )
        .withColumn("map_key", F.concat_ws("|", F.upper(F.trim(F.col("MarketCode"))), F.lower(F.trim(F.col("cbr_type")))))
        .withColumn("kafka_topics", mapping_expr[F.col("map_key")])
        .select(F.col("MDMKey"), F.col("FinalJSON"), F.col("kafka_topics"))
        .cache()
    )

    try:
        if backup_df.isEmpty():
            print(f"No backup records found for MarketCode={market_code}, MDMKey={mdm_key}")
            return

        kafka_brokers = get_env_config("target_kafka.kafka_brokers")

        kafka_df = (
            backup_df
            .where(
                F.col("kafka_topics").isNotNull() &
                F.col("FinalJSON").isNotNull() &
                (F.length(F.col("FinalJSON")) > 0)
            )
            .select(
                F.col("MDMKey").alias("key"),
                F.col("FinalJSON").alias("value"),
                # 一个备份记录可以配置多个目标 topic，逗号分隔后展开成多条 Kafka 消息。
                F.explode(F.split(F.col("kafka_topics"), ",")).alias("topic"),
            )
            .withColumn("topic", F.trim(F.col("topic")))
            .where(F.length(F.col("topic")) > 0)
            .dropDuplicates(["key", "value", "topic"])
        )

        if kafka_df.isEmpty():
            print("No valid Kafka messages to send")
            return

        message_count = kafka_df.count()
        print(f"Sending {message_count} messages to Kafka for MarketCode={market_code}, MDMKey={mdm_key}")

        (
            kafka_df
            .write
            .format("kafka")
            .option("kafka.bootstrap.servers", kafka_brokers)
            .option("kafka.request.timeout.ms", "15000")
            .option("kafka.max.block.ms", "20000")
            .option("kafka.delivery.timeout.ms", "30000")
            .option("kafka.retries", "0")
            .option("kafka.compression.type", "snappy")
            .mode("append")
            .save()
        )

        print(f"Sent {message_count} messages successfully")
    finally:
        backup_df.unpersist()

In [0]:
market_code = dbutils.widgets.get("MarketCode")
mdm_key = dbutils.widgets.get("MDMKey")
print(f"MarketCode: {market_code}, MDMKey: {mdm_key}")

step_name = "retry_backup_cbr_to_kafka"
step_num = "05"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    retry_backup_cbr_to_kafka(market_code, mdm_key)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=f"{market_code}_{mdm_key}",
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )